# 03 — Bronze → Silver: meteo

Junta los parquets diarios cacheados, agrega a semanal y limpia nombres de distrito para poder cruzar después con socio/epi. Salida: `data/silver/meteo_semanal_distrital.csv`.

In [1]:
import sys
from pathlib import Path

In [2]:
import pandas as pd
from denguekay.config.paths import REFERENCE, SILVER
from denguekay.data.processing.weekly_aggregation import (
    load_daily_from_cache, aggregate_to_weekly, clean_names_for_join,
)

distritos = pd.read_csv(REFERENCE / "distritos_piura_coords.csv")

In [3]:
diario = load_daily_from_cache(expected_districts=len(distritos))
diario.groupby("id_distrito").size().describe()

65 archivos


count      65.0
mean     3287.0
std         0.0
min      3287.0
25%      3287.0
50%      3287.0
75%      3287.0
max      3287.0
dtype: float64

In [4]:
semanal = aggregate_to_weekly(diario, distritos)
semanal = clean_names_for_join(semanal)
semanal.head()

,provincia,distrito,lat,lon,anio,semana,semana_inicio,temp_media,temp_max,temp_min,precip_total_mm,lluvia_total_mm,hum_rel_media,viento_max,radiacion_total,et0_total,distrito_key
13132,Paita,Amotape,-4.836343,-81.006897,2017,1,2017-01-01,25.585714,33.3,21.5,6.5,6.5,66.000000,28.4,142.70,34.03,AMOTAPE
13133,Paita,Amotape,-4.836343,-81.006897,2017,2,2017-01-08,25.357143,32.8,22.1,8.7,8.7,71.285714,26.9,130.27,30.07,AMOTAPE
13134,Paita,Amotape,-4.836343,-81.006897,2017,3,2017-01-15,26.314286,33.5,22.5,5.2,5.2,68.714286,24.0,138.79,33.84,AMOTAPE
13135,Paita,Amotape,-4.836343,-81.006897,2017,4,2017-01-22,26.828571,33.5,22.9,7.8,7.8,68.285714,26.3,147.98,35.54,AMOTAPE
13136,Paita,Amotape,-4.836343,-81.006897,2017,5,2017-01-29,26.185714,32.1,23.2,96.8,96.8,80.285714,23.2,133.78,28.88,AMOTAPE


In [5]:
SILVER.mkdir(parents=True, exist_ok=True)
semanal.to_csv(SILVER / "meteo_semanal_distrital.csv", index=False)
print(semanal.shape)

(30485, 17)


## Estudio 06 · fase 1b · historia 2000-2016 (aparte)

Agrega a semanas el caché `cache_meteo_2000_2016/` (celda B del notebook 02) con la misma función que
2017-2025 y lo guarda en `data/silver/meteo_semanal_distrital_2000_2016.csv`. No toca
`meteo_semanal_distrital.csv` ni el integrado. El 2016-12-31 es sábado, así que la última semana queda completa
y empalma con la primera de 2017; el 2000-01-01 también es sábado, así que esa semana de un día se descarta.


In [ ]:
from denguekay.config.paths import CACHE_METEO_2000_2016, SILVER_METEO_2000_2016

diario_historia = load_daily_from_cache(expected_districts=len(distritos), cache_dir=CACHE_METEO_2000_2016)
semanal_historia = clean_names_for_join(aggregate_to_weekly(diario_historia, distritos))
semanal_historia.to_csv(SILVER_METEO_2000_2016, index=False)
print(semanal_historia.shape, semanal_historia.semana_inicio.min(), semanal_historia.semana_inicio.max())


## Estudio 06 · 1b′ · ERA5-Land 2000-2025 (humedad y temperatura, aparte)

Une los cachés de ERA5-Land 2000-2016 y 2017-2025 (celdas A2, A3 y B2 del notebook 02), agrega a semanas solo las
variables que ERA5-Land trae y lo guarda en `data/silver/meteo_humedad_era5_land_2000_2025.csv`. Una semana sin datos
queda vacía, no en 0. No toca `meteo_semanal_distrital.csv`, el integrado ni el gold. Los 16 distritos de la muestra
de A2 traen además columnas vacías (precipitación, viento...): no se usan.


In [ ]:
from denguekay.config.paths import (
    CACHE_METEO_ERA5_LAND_2000_2016, CACHE_METEO_ERA5_LAND_2017_2025, SILVER_METEO_ERA5_LAND_2000_2025,
)
from denguekay.data.ingestion.openmeteo import ERA5_LAND_DAILY_VARS
from denguekay.data.processing.weekly_aggregation import aggregate_available_to_weekly, load_daily_from_caches

diario_land = load_daily_from_caches([CACHE_METEO_ERA5_LAND_2000_2016, CACHE_METEO_ERA5_LAND_2017_2025],
                                     expected_districts=len(distritos))
semanal_land = clean_names_for_join(aggregate_available_to_weekly(diario_land, distritos, ERA5_LAND_DAILY_VARS))
semanas = semanal_land.groupby("id_distrito").semana_inicio.agg(["min", "max", "count"])
assert semanas["count"].nunique() == 1, "No todos los distritos tienen las mismas semanas"
assert not semanal_land.duplicated(["id_distrito", "semana_inicio"]).any()
print("Semanas vacías por variable:", semanal_land[["temp_media", "hum_rel_media"]].isna().sum().to_dict())
semanal_land.to_csv(SILVER_METEO_ERA5_LAND_2000_2025, index=False)
print(semanal_land.shape, semanal_land.semana_inicio.min(), semanal_land.semana_inicio.max())


## ERA5-Land 2000 → hoy (silver de la v3)

Une los cachés de ERA5-Land 2000-2016, 2017-2025, 2026 y la foto reciente más nueva (celda C2 del notebook 02) y
escribe `data/silver/meteo_humedad_era5_land_desde_2000.csv`, que leen `gold.py` (d3), `load_data` (columna
`temp_media_era5_land_c`) y el paquete de la v3. Solo semanas de 7 días; una semana sin dato queda vacía, no en 0. Si
todavía no existe, el código usa el silver congelado de la v3 (`meteo_humedad_era5_land_2000_2026.csv`, del Release).

In [ ]:
from denguekay.config.paths import (
    CACHE_METEO_ERA5_LAND_2026, CACHE_METEO_ERA5_LAND_RECENT, SILVER_METEO_ERA5_LAND,
)
from denguekay.data.processing.weekly_aggregation import latest_snapshot

cachés = [CACHE_METEO_ERA5_LAND_2000_2016, CACHE_METEO_ERA5_LAND_2017_2025, CACHE_METEO_ERA5_LAND_2026]
reciente = latest_snapshot(CACHE_METEO_ERA5_LAND_RECENT)
if reciente is not None:
    cachés.append(reciente)
diario_land_actual = load_daily_from_caches(cachés, expected_districts=len(distritos))
semanal_land_actual = clean_names_for_join(aggregate_available_to_weekly(diario_land_actual, distritos, ERA5_LAND_DAILY_VARS))
assert semanal_land_actual.groupby("id_distrito").semana_inicio.count().nunique() == 1, "No todos los distritos tienen las mismas semanas"
assert not semanal_land_actual.duplicated(["id_distrito", "semana_inicio"]).any()
semanal_land_actual.to_csv(SILVER_METEO_ERA5_LAND, index=False)
print(semanal_land_actual.shape, semanal_land_actual.semana_inicio.min(), semanal_land_actual.semana_inicio.max(), "|", reciente)